# C9-dimensionality-reduction — Session 2: PCA from Covariance to a Reusable NumPy Class

*One class session, roughly 90 minutes.*
This session builds on Session 1's centered-data SVD route, F6's spectral decomposition, and F5's sample-variance convention.

**This session:** derive the centered covariance eigenproblem from directional variance; prove why covariance eigenvectors and centered-data right singular vectors describe the same principal subspaces; distinguish sign ambiguity from repeated-eigenvalue rotation; implement a reusable `NumpyPCA` with explicit fit state; and connect transforms, inverse transforms, projectors, and reconstruction error.

The denominator is fixed throughout: sample covariance and sample variance both divide by $n-1$.
Changing the denominator rescales eigenvalues but not eigenvectors, yet mixing conventions makes reported variances inconsistent.


In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

## 1. One denominator, three views of variance

Let $X$ have $n\ge 2$ rows and $d$ columns, let $\mu$ be its column-mean vector, and set $X_c=X-\mu$.
The **sample covariance matrix** is
$$C=\frac{1}{n-1}X_c^{\mathsf T}X_c\in\mathbb R^{d\times d}. $$
Its diagonal entry $C_{jj}$ is feature $j$'s sample variance, and $C_{jk}$ measures how centered features $j$ and $k$ move together.
It is symmetric, and for every vector $a$,
$$a^{\mathsf T}Ca=\frac{\lVert X_ca\rVert_2^2}{n-1}\ge 0,$$
so $C$ is positive semidefinite even when $X_c$ is rank deficient.

The trace is the total sample variance by two routes:
$$\operatorname{tr}(C)=\sum_j\operatorname{Var}(X_{:,j})=\frac{\lVert X_c\rVert_F^2}{n-1}. $$
These identities are the fastest denominator audit in this unit.


In [ ]:
X_small = np.array([
    [4.0, 1.0, 2.0],
    [2.0, 3.0, 0.0],
    [0.0, 1.0, 2.0],
    [2.0, -1.0, 4.0],
])
mu_small = X_small.mean(axis=0)
Xc_small = X_small - mu_small
n_small = X_small.shape[0]
C_small = Xc_small.T @ Xc_small / (n_small - 1)

assert np.allclose(C_small, C_small.T, atol=ATOL, rtol=RTOL)
assert np.allclose(
    np.diag(C_small), X_small.var(axis=0, ddof=1),
    atol=ATOL, rtol=RTOL,
)
assert np.isclose(
    np.trace(C_small), (Xc_small * Xc_small).sum() / (n_small - 1),
    atol=ATOL, rtol=RTOL,
)
print("mean:", mu_small)
print("sample covariance:\n", np.round(C_small, 4))
print("total sample variance:", np.trace(C_small))

### Checkpoint 1

1. What are the shapes of $X_c$, $X_c^{\mathsf T}X_c$, and $C$?
2. Why can $C$ have a zero eigenvalue without any input being zero?
3. If someone divides $C$ by $n$ but reports component variances with $n-1$, which quantities remain geometrically correct and which reported numbers are inconsistent?


## 2. Directional variance becomes an eigenproblem

For a unit direction $u$, Session 1 defined the score vector $z=X_cu$.
Its sample variance is
$$\operatorname{Var}(z)=\frac{\lVert X_cu\rVert_2^2}{n-1}=u^{\mathsf T}Cu. $$
Thus PC1 solves $\max_{\lVert u\rVert=1}u^{\mathsf T}Cu$.

Here is a calculus-AB-compatible stationarity argument.
At a maximizing unit vector $u$, choose any $w\perp u$ and move along the unit sphere via $u(t)=(u+tw)/\lVert u+tw\rVert$.
The derivative at $t=0$ is $2w^{\mathsf T}Cu$, and it must vanish for every $w\perp u$.
Therefore $Cu$ has no component orthogonal to $u$: for some scalar $\lambda$,
$$Cu=\lambda u. $$
Multiplying on the left by $u^{\mathsf T}$ shows $\lambda=u^{\mathsf T}Cu$, the directional variance itself.
The spectral decomposition $C=Q\Lambda Q^{\mathsf T}$ then proves the maximum is the largest eigenvalue: if $u=Qa$ with $\sum_i a_i^2=1$,
$$u^{\mathsf T}Cu=\sum_i\lambda_i a_i^2\le\lambda_{\max}. $$
Subsequent PCs repeat the argument under orthogonality constraints, producing an orthonormal eigenbasis in descending eigenvalue order.


In [ ]:
evals_small, evecs_small = np.linalg.eigh(C_small)
order_small = np.argsort(evals_small)[::-1]
evals_small = evals_small[order_small]
evecs_small = evecs_small[:, order_small]
u1_small = evecs_small[:, 0]
lambda1_small = evals_small[0]

assert np.isclose(np.linalg.norm(u1_small), 1.0, atol=ATOL, rtol=RTOL)
assert np.allclose(
    C_small @ u1_small, lambda1_small * u1_small,
    atol=ATOL, rtol=RTOL,
)
assert np.isclose(
    ((Xc_small @ u1_small) ** 2).sum() / (n_small - 1),
    lambda1_small, atol=ATOL, rtol=RTOL,
)
print("descending component variances:", np.round(evals_small, 6))
print("PC1 directional variance:", np.round(lambda1_small, 6))

### Checkpoint 2

1. In the stationarity argument, why must $w$ be orthogonal to $u$?
2. Once $Cu=\lambda u$ and $\lVert u\rVert=1$, why is $\lambda$ exactly the score variance?
3. Explain why a negative covariance eigenvalue would contradict $a^{\mathsf T}Ca\ge0$.


## 3. Why the covariance and SVD routes agree

Take the centered-data SVD from Session 1:
$$X_c=U\Sigma V^{\mathsf T}. $$
Then
$$C=\frac{X_c^{\mathsf T}X_c}{n-1}=V\frac{\Sigma^{\mathsf T}\Sigma}{n-1}V^{\mathsf T}. $$
Therefore the covariance eigenvalues are $\sigma_i^2/(n-1)$ and its eigenvectors are the right singular directions, the rows of `Vt`.
This is an identity, not an approximate empirical resemblance.

The routes differ computationally.
Forming $C$ squares the condition number and creates a $d\times d$ matrix; direct SVD is often numerically safer.
For this lesson's from-scratch class, covariance plus `np.linalg.eigh` is valuable because it makes the derivation and full $d$-dimensional zero-eigenspace visible.
The class must still agree with the centered SVD on eigenvalues and identifiable subspaces.


In [ ]:
_, s_small, Vt_small = np.linalg.svd(Xc_small, full_matrices=False)
svd_variances_small = s_small**2 / (n_small - 1)

assert np.allclose(
    evals_small[:len(s_small)], svd_variances_small,
    atol=ATOL, rtol=RTOL,
)
for j in range(len(s_small)):
    assert np.isclose(
        abs(evecs_small[:, j] @ Vt_small[j]), 1.0,
        atol=ATOL, rtol=RTOL,
    )
print("eigh variances:", np.round(evals_small, 6))
print("SVD variances :", np.round(svd_variances_small, 6))

## 4. Sign ambiguity is one-dimensional; repeated eigenspaces can rotate

For a simple eigenvalue, an eigenvector is unique only up to sign, so $vv^{\mathsf T}=(-v)(-v)^{\mathsf T}$ is the sign-free one-dimensional projector.
For a repeated eigenvalue, the ambiguity is larger: any orthonormal basis of the repeated eigenspace is valid.
Comparing basis vector 1 to basis vector 1 is then meaningless because another routine may rotate or swap the basis.

If the rows of $Q$ form an orthonormal basis of the target subspace, compare
$$P=Q^{\mathsf T}Q. $$
Two bases describe the same subspace exactly when their projectors agree.
This rule covers signs, swaps, and arbitrary rotations inside a repeated block.
It does **not** license mixing across unequal eigenvalues.


In [ ]:
X_repeat = np.array([
    [np.sqrt(2.0), 0.0, 0.0],
    [-np.sqrt(2.0), 0.0, 0.0],
    [0.0, np.sqrt(2.0), 0.0],
    [0.0, -np.sqrt(2.0), 0.0],
])
C_repeat = X_repeat.T @ X_repeat / (len(X_repeat) - 1)
evals_repeat, evecs_repeat = np.linalg.eigh(C_repeat)
Q_eigh = evecs_repeat[:, np.argsort(evals_repeat)[::-1][:2]].T
theta = 0.37
R2 = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta), np.cos(theta)],
])
Q_rotated = R2 @ Q_eigh
P_eigh = Q_eigh.T @ Q_eigh
P_rotated = Q_rotated.T @ Q_rotated

assert not np.allclose(Q_eigh, Q_rotated, atol=ATOL, rtol=RTOL)
assert np.allclose(P_eigh, P_rotated, atol=ATOL, rtol=RTOL)
assert np.allclose(P_eigh @ P_eigh, P_eigh, atol=ATOL, rtol=RTOL)
print("repeated spectrum:", evals_repeat[::-1])
print("basis gap:", np.linalg.norm(Q_eigh - Q_rotated))
print("projector gap:", np.linalg.norm(P_eigh - P_rotated))

### Checkpoint 3

1. Why does taking absolute dot products solve a sign flip but not a general rotation of a two-dimensional repeated eigenspace?
2. If `components` has orthonormal rows, prove $P=\text{components.T @ components}$ is symmetric and idempotent.
3. Which is invariant under a change of basis inside the retained subspace: the individual score columns, their joint Euclidean norm, or both?


## 5. The reusable `NumpyPCA` fit-state contract

A reusable estimator separates **learning state** from **using state**.
`fit(X)` validates and centers the training data, performs one symmetric eigendecomposition, stores what later calls need, and returns `self`.
`transform(X_new)` must reuse the training mean and directions; recomputing a mean on new data changes the coordinate system and is a quiet bug.

After fitting $k$ components, this course requires:

- `mean_`: shape `(d,)`;
- `components_`: shape `(k, d)`, orthonormal rows in descending explained-variance order;
- `explained_variance_`: shape `(k,)`, covariance eigenvalues $\sigma_i^2/(n-1)$;
- `explained_variance_ratio_`: shape `(k,)`, each retained eigenvalue divided by the sum of **all** covariance eigenvalues.

If all data rows are identical, total variance is zero and the ratio vector is defined as zeros rather than `0/0`.
The class uses NumPy only; sklearn and scipy PCA are outside this task's contract.


In [ ]:
class NumpyPCA:
    def __init__(self, n_components):
        self.n_components = n_components

    @staticmethod
    def _as_finite_matrix(X, *, min_rows):
        X = np.asarray(X)
        if (X.ndim != 2 or X.shape[0] < min_rows or X.shape[1] == 0
                or not np.issubdtype(X.dtype, np.number)):
            raise ValueError("X must be a finite numeric matrix")
        X = X.astype(float, copy=False)
        if not np.isfinite(X).all():
            raise ValueError("X must be finite")
        return X

    def _require_fitted(self):
        if not hasattr(self, "components_"):
            raise ValueError("fit must be called first")

    def fit(self, X):
        X = self._as_finite_matrix(X, min_rows=2)
        d = X.shape[1]
        if (isinstance(self.n_components, bool)
                or not isinstance(self.n_components, (int, np.integer))
                or not 1 <= int(self.n_components) <= d):
            raise ValueError("n_components must be an integer in [1, d]")
        k = int(self.n_components)
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_
        covariance = Xc.T @ Xc / (X.shape[0] - 1)
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = np.maximum(eigenvalues[order], 0.0)
        eigenvectors = eigenvectors[:, order]
        self.components_ = eigenvectors[:, :k].T
        self.explained_variance_ = eigenvalues[:k]
        total = eigenvalues.sum()
        self.explained_variance_ratio_ = (
            np.zeros(k) if total == 0.0 else eigenvalues[:k] / total
        )
        return self

    def transform(self, X):
        self._require_fitted()
        X = self._as_finite_matrix(X, min_rows=1)
        if X.shape[1] != self.mean_.shape[0]:
            raise ValueError("feature count does not match fit data")
        return (X - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        self._require_fitted()
        Z = self._as_finite_matrix(Z, min_rows=1)
        if Z.shape[1] != self.components_.shape[0]:
            raise ValueError("score count does not match retained components")
        return Z @ self.components_ + self.mean_

Three details matter.
First, `np.linalg.eigh` returns eigenvalues ascending, so values and eigenvectors are reordered together.
Second, tiny negative eigenvalues from roundoff are clipped only after decomposition; a true covariance matrix is positive semidefinite.
Third, the explained-ratio denominator uses the full spectrum, not only the retained $k$ entries.
Otherwise every reduced model would falsely claim its retained ratios sum to one.

### Checkpoint 4

1. Why must `fit` return `self`?
2. Why is `components_` stored with directions as rows when `eigh` returns them as columns?
3. What quiet error results from dividing by `explained_variance_.sum()` when $k<d$?
4. Which state must a second call to `fit` replace?


## 6. Transform, inverse transform, and the projector

For fitted row-components $Q$,
$$Z=(X-\mu)Q^{\mathsf T}$$
contains the retained scores, and
$$\widehat X=ZQ+\mu=(X-\mu)Q^{\mathsf T}Q+\mu$$
is the reconstruction.
Thus `inverse_transform(transform(X))` is centered projection by $P=Q^{\mathsf T}Q$, followed by adding the training mean back.
With all $d$ components, $P=I$ and reconstruction is exact up to floating-point error.
With $k<d$, the squared Frobenius reconstruction error is
$$\lVert X-\widehat X\rVert_F^2=(n-1)\sum_{j>k}\lambda_j=\sum_{j>k}\sigma_j^2. $$


In [ ]:
pca2 = NumpyPCA(2)
Z_small = pca2.fit_transform(X_small)
Xhat_small = pca2.inverse_transform(Z_small)
P_small = pca2.components_.T @ pca2.components_
Xhat_projector = (X_small - pca2.mean_) @ P_small + pca2.mean_

assert np.allclose(Xhat_small, Xhat_projector, atol=ATOL, rtol=RTOL)
assert np.allclose(P_small @ P_small, P_small, atol=ATOL, rtol=RTOL)
full_evals = np.linalg.eigvalsh(C_small)[::-1]
error2 = ((X_small - Xhat_small) ** 2).sum()
tail2 = (n_small - 1) * full_evals[2:].sum()
assert np.isclose(error2, tail2, atol=ATOL, rtol=RTOL)

pca_full = NumpyPCA(X_small.shape[1])
X_full_hat = pca_full.inverse_transform(pca_full.fit_transform(X_small))
assert np.allclose(X_full_hat, X_small, atol=ATOL, rtol=RTOL)
print("scores shape:", Z_small.shape)
print("rank-2 squared error / spectral tail:", error2, tail2)

### Checkpoint 5

1. Derive the projector form of reconstruction by substituting the transform formula into inverse transform.
2. Why must inverse transform add the **training** mean rather than the score mean?
3. A rank-deficient centered matrix has rank $r$.
What reconstruction error should $k=r$ achieve, and what happens to later explained variances?


## 7. Worked exam-style example: certify a PCA implementation

**Problem.** Fit two components to the four-row dataset from Section 1.
Report the mean, component variances, explained ratios, score shape, projector, and squared reconstruction error.
Certify the result independently through the centered-data SVD and the covariance eigendecomposition.
Use `ATOL = 1e-10`, `RTOL = 0.0`; do not compare raw signed component rows.

**Solution route.** Fit the class, build its projector, then compare eigenvalues and projectors rather than individual direction signs.


In [ ]:
worked = NumpyPCA(2).fit(X_small)
worked_scores = worked.transform(X_small)
worked_hat = worked.inverse_transform(worked_scores)
worked_projector = worked.components_.T @ worked.components_

_, s_worked, Vt_worked = np.linalg.svd(Xc_small, full_matrices=False)
svd_projector = Vt_worked[:2].T @ Vt_worked[:2]
assert np.allclose(
    worked.explained_variance_, s_worked[:2]**2 / (n_small - 1),
    atol=ATOL, rtol=RTOL,
)
assert np.allclose(worked_projector, svd_projector, atol=ATOL, rtol=RTOL)
assert np.allclose(
    worked_scores @ worked_scores.T,
    (Xc_small @ worked_projector) @ (Xc_small @ worked_projector).T,
    atol=ATOL, rtol=RTOL,
)
print("mean:", worked.mean_)
print("component variances:", np.round(worked.explained_variance_, 6))
print("explained ratios:", np.round(worked.explained_variance_ratio_, 6))
print("score shape:", worked_scores.shape)
print("squared reconstruction error:", ((X_small - worked_hat) ** 2).sum())

## 8. Common pitfalls

**Pitfall 1 — inconsistent denominators (quiet).**
Using `np.cov` defaults without checking orientation or mixing $n$ and $n-1$ leaves directions plausible while reported variances fail anchors.
Fix: form `Xc.T @ Xc / (n - 1)` explicitly and compare its diagonal with `X.var(axis=0, ddof=1)`.

**Pitfall 2 — sorting eigenvalues without eigenvectors (quiet, catastrophic).**
`np.sort(eigenvalues)[::-1]` alone destroys the eigenpair correspondence.
Fix: compute one `order` and apply it to both arrays.

**Pitfall 3 — testing repeated eigenvectors row by row (false failure).**
A rotated basis of a repeated eigenspace is equally correct.
Fix: compare `Q.T @ Q` projectors with explicit tolerances.

**Pitfall 4 — fitting during transform (quiet leakage).**
Subtracting `X_new.mean(axis=0)` or recomputing components makes train and new rows live in different coordinate systems.
Fix: transform uses only `mean_` and `components_` learned by fit.

**Pitfall 5 — retained-only ratio denominator (quiet).**
Dividing by the retained eigenvalue sum makes every truncated model report total ratio one.
Fix: divide retained eigenvalues by the full covariance trace.

### Checkpoint 6

1. Give one invariant that catches each of the five pitfalls.
2. Which pitfalls can pass all shape checks?
3. Why is a green sklearn comparison not a substitute for understanding these contracts?


## Exam connections and going deeper

Round 1 questions often hide the eigenproblem inside a variance phrase, ask which denominator matches a stated convention, or present two different valid bases for the same repeated eigenspace.
Coding versions grade state and data flow: `transform` must use fitted state, ratios must use the full variance, and reconstruction must follow the retained projector.
The safest response is a chain of independently checkable invariants, not a black-box library answer.

Going deeper, numerical linear algebra studies why direct SVD is safer than explicitly forming $X_c^{\mathsf T}X_c$, and C10-competition-craft will reuse the fit/transform boundary to prevent validation leakage.
Neither topic is assumed in this session's graded work.


## Checkpoint answers

<details><summary><b>Checkpoint 1</b></summary>

1. $(n,d)$, $(d,d)$, and $(d,d)$.
2. Linear dependence among centered columns gives a nonzero $a$ with $X_ca=0$, hence $Ca=0$.
3. Directions are unchanged by a common positive rescaling, but covariance eigenvalues and component variances disagree by the factor $n/(n-1)$.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. It is a tangent direction to the unit sphere, so the perturbation respects the constraint to first order.
2. $u^{\mathsf T}Cu=u^{\mathsf T}(\lambda u)=\lambda$.
3. Its eigenvector $a$ would give $a^{\mathsf T}Ca=\lambda\lVert a\rVert^2<0$.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Absolute dot products identify one axis, but a rotated repeated basis mixes several valid axes.
2. $P^{\mathsf T}=P$ and $P^2=Q^{\mathsf T}(QQ^{\mathsf T})Q=P$ because $QQ^{\mathsf T}=I$.
3. Individual score columns rotate; their joint norm and the reconstructed projection are invariant.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. It enables the standard `NumpyPCA(k).fit(X)` chain and makes fit-state identity testable.
2. Then rows index components and `(X - mean_) @ components_.T` produces `(n,k)` scores.
3. Retained ratios falsely sum to one and overstate preserved variance.
4. All four learned arrays; transform after refit must use the new mean and directions.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Substitute $Z=(X-\mu)Q^{\mathsf T}$ into $ZQ+\mu$.
2. Scores are coordinates around the fitted origin; adding another origin changes the data space.
3. Zero error up to roundoff; all later explained variances are zero.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. Diagonal-vs-`ddof=1`; eigen-residual pairing; projector equality; a shifted-new-data transform anchor; ratio sum against full trace.
2. All five can: they are semantic or numerical, not shape errors.
3. A library match does not establish the denominator, fitted-state data flow, or degenerate-subspace comparison the task asks you to explain.

</details>